# Preparing Document for Data generation
- This notebook will show you how to do document parsing
- Document Chunking
- And finally mixing it with user QNA to  create seed examples

## Install SDG

```bash 
pip install sdg-hub[examples]
```

In [ ]:
from datetime import datetime
import os

now = datetime.now()
timestamp = now.strftime('%Y%m%d-%H%M%S')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [ ]:
force_ascii = True

## Select a document

### 2024 IBM Annual Report

In [ ]:
data_dir = 'document_collection/ibm-annual-report'
preprocess_pdf = True
docling_config = "docling_v2_config.yaml"
list_md_files = [f"{data_dir}/ibm-annual-report-2024.md"]
user_config_path = f'{data_dir}/qna.yaml'

### Teigaku Genzei

In [ ]:
data_dir = 'document_collection/teigaku-genzei'
preprocess_pdf = True
docling_config = "docling_v2_config.yaml"
list_md_files = [f"{data_dir}/0024001-021.md"]
user_config_path = f'{data_dir}/qna_ja.yaml'

### IBM Newsroom

In [ ]:
data_dir = 'document_collection/ibm-newsroom'
preprocess_pdf = False
# docling_config = "docling_v2_config.yaml"
# list_md_files = [f"{data_dir}/qs2-riken.md", f"{data_dir}/qs1-utokyo.md", f"{data_dir}/jica.md"]
list_md_files = [f"{data_dir}/qs2-riken.md"]
user_config_path = f'{data_dir}/qna.yaml'

### IBM Newsroom (English)

In [ ]:
data_dir = 'document_collection/ibm-newsroom-en'
preprocess_pdf = False
# docling_config = "docling_v2_config.yaml"
list_md_files = [f"{data_dir}/IBM_Think_2025.md"]
user_config_path = f'{data_dir}/qna_Think_2025.yaml'

### Nencho (WIP)

In [ ]:
data_dir = 'document_collection/nencho'
preprocess_pdf = False
docling_config = "docling_v2_config-no_ocr.yaml"
list_md_files = [f"{data_dir}/nencho.md"]  # NOTE this file is created by concatenating ??.md after manual cleansing
user_config_path = f'{data_dir}/qna.yaml'

## Initialize common variables

In [ ]:
data_name = os.path.basename(data_dir)

seed_data_path = f"{data_dir}/seed_data_{data_name}.jsonl"
# seed_data_path = f"{data_dir}/seed_data_{data_name}_{timestamp}.jsonl"

## Pre-process PDF documents using docling v2

In [ ]:
if preprocess_pdf:
    # !OMP_NUM_THREADS=32 mamba run -n docling python docparser_v2.py --input-dir {data_dir} --output-dir {data_dir} --c {docling_config}
    !python docparser_v2.py --input-dir {data_dir} --output-dir {data_dir} -c {docling_config}

## Create Seed Examples

In [ ]:
from knowledge_utils import DocProcessor

dp = DocProcessor(data_dir, user_config_path=user_config_path)

### Using markdown file
# Note: For now v2 json is not supported
seed_data = dp.get_processed_markdown_dataset(list_md_files)

seed_data.to_json(seed_data_path, orient='records', lines=True, force_ascii=force_ascii)